In [5]:
import json
import os

from dotenv import load_dotenv
import httpx
import openai
from openai.types.chat import ChatCompletionMessage

load_dotenv()

MODEL = "gpt-4o-mini"
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

SYSTEM_PROMPT = """
You are a personalized movie recommendation chatbot backed by the Nomad Movies API. Your job is to help users discover films while remembering what they share across the conversation.

Memory and personalization (use the full message history in this chat):
- Track genres, directors, actors, eras, or moods the user says they like or dislike. Refer to these when suggesting or filtering titles.
- Remember titles the user says they have already watched (or want to skip). Do not recommend those again unless they explicitly ask for similar films or a re-watch.
- When recommending, tailor picks to stated tastes and prior choices in the thread; briefly explain why each suggestion fits them.

Tools:
- get_popular_movies — list popular movies (GET /movies). Use when the user asks what is popular or trending, or as a pool to narrow for personalized picks.
- get_movie_details — details for one movie by numeric id (GET /movies/:id). Use when the user asks about a specific movie by ID, or to inspect genre/plot for recommendations.
- get_movie_credits — cast and crew (GET /movies/:id/credits). Use when the user asks who stars, who directed, or credits for a movie ID.

Behavior:
- Call the appropriate tool to fetch data. After tool results, answer in natural language using that data.
- Match the user's language (e.g. Korean if they wrote in Korean).
- If preferences or watch history are unclear, ask a short clarifying question before recommending.
"""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Fetch popular movies from the /movies endpoint.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Fetch movie information for a given movie id.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "Movie id."}},
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Fetch cast and crew for a given movie id.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "Movie id."}},
                "required": ["id"],
            },
        },
    },
]


def get_popular_movies() -> object:
    r = httpx.get(f"{BASE_URL}/movies", timeout=5.0)
    r.raise_for_status()
    return r.json()


def get_movie_details(movie_id: int) -> object:
    r = httpx.get(f"{BASE_URL}/movies/{movie_id}", timeout=5.0)
    r.raise_for_status()
    return r.json()


def get_movie_credits(movie_id: int) -> object:
    r = httpx.get(f"{BASE_URL}/movies/{movie_id}/credits", timeout=5.0)
    r.raise_for_status()
    return r.json()


def execute_tool(name: str, arguments_json: str) -> str:
    try:
        args = json.loads(arguments_json)
        if name == "get_popular_movies":
            data = get_popular_movies()
        elif name == "get_movie_details":
            data = get_movie_details(int(args["id"]))
        elif name == "get_movie_credits":
            data = get_movie_credits(int(args["id"]))
        else:
            return json.dumps({"error": f"unknown tool: {name}"})
        return json.dumps(data)
    except Exception as e:
        return json.dumps({"error": type(e).__name__, "message": str(e)})


def _assistant_message_dict(msg: ChatCompletionMessage) -> dict:
    d: dict = {"role": "assistant"}
    if msg.content:
        d["content"] = msg.content
    if msg.tool_calls:
        d["tool_calls"] = [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in msg.tool_calls
        ]
    return d


def run_agent(user_content: str, messages: list | None = None) -> str:
    client = openai.OpenAI()
    if messages is None:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.append({"role": "user", "content": user_content})
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
    )
    msg = resp.choices[0].message
    if not msg.tool_calls:
        text = msg.content or ""
        messages.append({"role": "assistant", "content": text})
        print(f"AI: {text}")
        return text
    messages.append(_assistant_message_dict(msg))
    for tc in msg.tool_calls:
        out = execute_tool(tc.function.name, tc.function.arguments)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
        # print(f"- [tool] {tc.function.name} {tc.function.arguments}")
    resp2 = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="none",
    )
    text2 = resp2.choices[0].message.content or ""
    messages.append({"role": "assistant", "content": text2})
    print(f"AI: {text2}")
    return text2

In [6]:
# Run the previous cell first. History is kept in `messages`.
# End with: quit / exit / q

messages = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Movie Expert — type 'quit' to exit.\n")

while True:
    print("=" * 60)
    user_message = input("You: ").strip()
    print(f"User: {user_message}")
    if user_message.lower() in ("quit", "exit", "q"):
        print("Bye.")
        break
    if not user_message:
        continue
    run_agent(user_message, messages=messages)

Movie Expert — type 'quit' to exit.

User: 나는 SF 영화를 좋아해
AI: SF 영화를 좋아하신다니 멋지네요! 최근에 본 SF 영화가 있나요? 아니면 특별히 좋아하는 감독이나 배우가 있으신가요? 그런 정보를 알면 더 맞춤 추천을 드릴 수 있을 것 같아요.
User: 인셉션이랑 인터스텔라는 이미 봤어
AI: "인셉션"과 "인터스텔라"를 이미 보셨군요! 두 영화 모두 놀라운 상상력과 깊이 있는 스토리로 유명하죠. 그렇다면 다음으로 추천드릴 SF 영화 몇 편을 소개할게요.

1. **블레이드 러너 2049** - 원작인 "블레이드 러너"의 후속작으로, 뛰어난 비주얼과 서사로 인해 강한 몰입감을 제공합니다. 인간과 안드로이드의 경계를 탐구하며, 깊은 철학적 질문을 던지는 점이 매력적입니다.

2. **스칼렛** - 포스트 아포칼립틱 환경에서의 생존과 인간의 본성을 탐구하는 이야기입니다. 독특한 비주얼과 강렬한 감정선이 돋보입니다.

3. **테넷** - 크리스토퍼 놀란 감독의 또 다른 작품으로, 시간과 현실의 개념을 뒤틀어 놓는 이야기입니다. 복잡한 구성과 긴장감 넘치는 액션이 특징입니다.

이 영화들 중 어떤 것이 궁금한가요? 혹시 다른 추천을 원하시는 방향이 있으면 알려주세요!
User: 오늘 밤에 뭐 볼지 추천해 줄래?
AI: 오늘 밤에 볼 만한 SF 영화 몇 편을 추천해드릴게요!

1. **Project Hail Mary** (2026년 3월 개봉 예정)  
   - 설명: 과학 선생님이 우주선에서 깨어나 자신의 기억을 되찾아가는 과정에서, 태양의 멸망을 막기 위한 미션과 예상치 못한 우정이 얽히는 이야기입니다. 매력적인 설정과 서스펜스가 가득한 SF입니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/yihdXomYb5kTeSivtFndMy5iDmf.jpg)

2. **Hoppers** (2026년 3월 4일 개봉 예정)  
   - 설명: 인간의 의식을 로봇 동물로 옮기는 기술이 발견되고,